In [1]:
! pip install langchain langchain-google-genai langchain-community langgraph python-dotenv faiss-cpu pypdf

   ---------------------------------------- 0.0/16.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.6 MB 1.8 MB/s eta 0:00:09
   - -------------------------------------- 0.8/16.6 MB 1.6 MB/s eta 0:00:11
   -- ------------------------------------- 1.0/16.6 MB 1.5 MB/s eta 0:00:11
   --- ------------------------------------ 1.6/16.6 MB 1.6 MB/s eta 0:00:10
   ---- ----------------------------------- 1.8/16.6 MB 1.5 MB/s eta 0:00:10
   ----- ---------------------------------- 2.1/16.6 MB 1.6 MB/s eta 0:00:10
   ----- ---------------------------------- 2.4/16.6 MB 1.5 MB/s eta 0:00:10
   ------ --------------------------------- 2.6/16.6 MB 1.5 MB/s eta 0:00:10
   ------ --------------------------------- 2.9/16.6 MB 1.5 MB/s eta 0:00:10
   -------- ------------------------------- 3.4/16.6 MB 1.5 MB/s eta 0:00:09
   --------- ------------------------------ 3.9/16.6 MB 1.6 MB/s eta 0:00:09
   --------- 


[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.documents import Document
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing import TypedDict, Annotated, Optional
import os
import sqlite3
import tempfile
import requests
load_dotenv()

## RAG Chatbot — Gemini + FAISS

Follows the `langraph_rag_backend.py` pattern from campusx-official/chatbot-in-langgraph, but Gemini-only (`ChatGoogleGenerativeAI` + `GoogleGenerativeAIEmbeddings`):

1. Upload/ingest a **PDF** → chunk → **FAISS** index (kept per thread)
2. Tools: Tavily search, calculator, stock price, and `rag_tool` (retrieval over the uploaded doc)
3. Graph: `chat_node ⇄ tools` with **SqliteSaver** memory per thread
4. LangSmith tracing runs automatically via the env vars in `.env`

In [ ]:
# 1. LLM + embeddings (Gemini only)
llm = ChatGoogleGenerativeAI(
    model=os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite"),
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.7,
    max_tokens=8192,
)
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY"),
)

In [ ]:
# 2. Per-thread PDF store (FAISS retrievers)
_THREAD_RETRIEVERS = {}
_THREAD_METADATA = {}


def _get_retriever(thread_id):
    """Fetch the retriever for a thread if available."""
    if thread_id and thread_id in _THREAD_RETRIEVERS:
        return _THREAD_RETRIEVERS[thread_id]
    return None


def ingest_pdf(file_bytes: bytes, thread_id: str, filename: Optional[str] = None) -> dict:
    """Build a FAISS retriever for the uploaded PDF and store it for the thread."""
    if not file_bytes:
        raise ValueError("No bytes received for ingestion.")

    temp_path = None
    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as temp_file:
            temp_file.write(file_bytes)
            temp_path = temp_file.name

        docs = PyPDFLoader(temp_path).load()

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000, chunk_overlap=200, separators=["\n\n", "\n", " ", ""]
        )
        chunks = splitter.split_documents(docs)

        vector_store = FAISS.from_documents(chunks, embeddings)
        retriever = vector_store.as_retriever(
            search_type="similarity", search_kwargs={"k": 4}
        )

        meta = {
            "filename": filename or os.path.basename(temp_path),
            "documents": len(docs),
            "chunks": len(chunks),
        }
        _THREAD_RETRIEVERS[str(thread_id)] = retriever
        _THREAD_METADATA[str(thread_id)] = meta
        return meta
    finally:
        if temp_path:
            try:
                os.remove(temp_path)
            except OSError:
                pass

In [ ]:
# 3. Tools
search_tool = TavilySearchResults(max_results=5)


@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers.
    Supported operations: add, sub, mul, div
    """
    try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "sub":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        elif operation == "div":
            if second_num == 0:
                return {"error": "Division by zero is not allowed"}
            result = first_num / second_num
        else:
            return {"error": f"Unsupported operation '{operation}'"}

        return {
            "first_num": first_num,
            "second_num": second_num,
            "operation": operation,
            "result": result,
        }
    except Exception as e:
        return {"error": str(e)}


@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch the latest stock price for a given symbol (e.g. 'AAPL', 'TSLA')
    using the Alpha Vantage API.
    """
    api_key = os.getenv("ALPHA_VANTAGE_API_KEY")
    if not api_key:
        return {"error": "Set ALPHA_VANTAGE_API_KEY in .env to use this tool"}
    url = (
        "https://www.alphavantage.co/query"
        f"?function=GLOBAL_QUOTE&symbol={symbol}&apikey={api_key}"
    )
    r = requests.get(url, timeout=15)
    return r.json()


@tool
def rag_tool(query: str, thread_id: Optional[str] = None) -> dict:
    """
    Retrieve relevant information from the uploaded PDF for this chat thread.
    Always include the thread_id when calling this tool.
    """
    retriever = _get_retriever(thread_id)
    if retriever is None:
        return {
            "error": "No document indexed for this chat. Upload a PDF first.",
            "query": query,
        }

    result = retriever.invoke(query)
    context = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]

    return {
        "query": query,
        "context": context,
        "metadata": metadata,
        "source_file": _THREAD_METADATA.get(str(thread_id), {}).get("filename"),
    }


tools = [search_tool, get_stock_price, calculator, rag_tool]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
# 4. State + nodes
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


def chat_node(state: ChatState, config=None):
    """LLM node that may answer or request a tool call."""
    thread_id = None
    if config and isinstance(config, dict):
        thread_id = config.get("configurable", {}).get("thread_id")

    system_message = SystemMessage(
        content=(
            "You are a helpful assistant. For questions about the uploaded document, call "
            "the `rag_tool` and include the thread_id "
            f"`{thread_id}`. You can also use the web search, stock price, and "
            "calculator tools when helpful. If no document is available, ask the user "
            "to ingest one first."
        )
    )

    messages = [system_message, *state["messages"]]
    response = llm_with_tools.invoke(messages, config=config)
    return {"messages": [response]}


tool_node = ToolNode(tools)


# 5. Checkpointer (SQLite memory)
conn = sqlite3.connect(database="rag_chatbot.db", check_same_thread=False)
checkpointer = SqliteSaver(conn=conn)


# 6. Graph
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node", tools_condition)
graph.add_edge("tools", "chat_node")

chatbot = graph.compile(checkpointer=checkpointer)

In [ ]:
# 7. Helpers
def retrieve_all_threads():
    all_threads = set()
    for checkpoint in checkpointer.list(None):
        all_threads.add(checkpoint.config["configurable"]["thread_id"])
    return list(all_threads)


def thread_has_document(thread_id: str) -> bool:
    return str(thread_id) in _THREAD_RETRIEVERS


def thread_document_metadata(thread_id: str) -> dict:
    return _THREAD_METADATA.get(str(thread_id), {})

## Smoke test with a synthetic document (no PDF needed)

In [ ]:
# 8a. Index a synthetic document to smoke-test the retrieval loop
thread_id = "rag-demo"

sample_text = (
    "Orbital Space Storage (OSS) is a fictional cold-storage service that keeps encrypted data "
    "vaults in low-Earth orbit.\n\n"
    "Data retention: All customer data is retained for 7 years minimum by default. Customers may "
    "purchase a 99-year deep-archive tier. Deletion requests are honored within 30 days after "
    "cryptographic erasure of vault indexes.\n\n"
    "Encryption: Every vault uses AES-256 at rest and TLS 1.3 in transit. Master keys are rotated "
    "every 90 days and stored in the fictional OrbitalKMS service.\n\n"
    "Pricing: The standard tier costs $12 per terabyte-month. The deep-archive tier costs $3 per "
    "terabyte-month with a 99.9% availability guarantee."
)

docs = [Document(page_content=p, metadata={"source": "oss-spec"}) for p in sample_text.split("\n\n")]
chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100).split_documents(docs)
vs = FAISS.from_documents(chunks, embeddings)
_THREAD_RETRIEVERS[thread_id] = vs.as_retriever(search_kwargs={"k": 4})
_THREAD_METADATA[thread_id] = {"filename": "oss-spec", "documents": len(docs), "chunks": len(chunks)}
print(f"indexed: {_THREAD_METADATA[thread_id]}")

q1 = chatbot.invoke(
    {"messages": [HumanMessage(content="What is the data retention policy in the document?")]},
    {"configurable": {"thread_id": thread_id}},
)
print(q1["messages"][-1].content)

q2 = chatbot.invoke(
    {"messages": [HumanMessage(content="And how much does the deep-archive tier cost per terabyte-month?")]},
    {"configurable": {"thread_id": thread_id}},
)
print(q2["messages"][-1].content)

## Ingest your own PDF

Set `pdf_path` to any local PDF, then ask questions about it — `rag_tool` retrieves from the FAISS index while the graph keeps conversation memory per thread (`rag_chatbot.db`).

In [ ]:
# 8b. Real PDF ingestion + Q&A
pdf_path = r"C:\path\to\your.pdf"

with open(pdf_path, "rb") as f:
    info = ingest_pdf(f.read(), thread_id="pdf-demo", filename=os.path.basename(pdf_path))
print("indexed:", info)

answer = chatbot.invoke(
    {"messages": [HumanMessage(content="Summarize the document in 5 bullet points.")]},
    {"configurable": {"thread_id": "pdf-demo"}},
)
print(answer["messages"][-1].content)